<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/test_superconductivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
FILE_PATH = 'D:\HCMUT\AIO\conquer1\AIO-Conquer-Data\Data\dataset\superconductivity_processed.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
C:\Users\ngami\AppData\Local\Temp\ipykernel_32880\1805616756.py:1: SyntaxWarning: invalid escape sequence '\H'
  FILE_PATH = 'D:\HCMUT\AIO\conquer1\AIO-Conquer-Data\Data\dataset\superconductivity_processed.csv'


Đã tải dữ liệu thành công! Kích thước: (21197, 82)
<class 'pandas.DataFrame'>
RangeIndex: 21197 entries, 0 to 21196
Data columns (total 82 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   number_of_elements               21197 non-null  int64  
 1   mean_atomic_mass                 21197 non-null  float64
 2   wtd_mean_atomic_mass             21197 non-null  float64
 3   gmean_atomic_mass                21197 non-null  float64
 4   wtd_gmean_atomic_mass            21197 non-null  float64
 5   entropy_atomic_mass              21197 non-null  float64
 6   wtd_entropy_atomic_mass          21197 non-null  float64
 7   range_atomic_mass                21197 non-null  float64
 8   wtd_range_atomic_mass            21197 non-null  float64
 9   std_atomic_mass                  21197 non-null  float64
 10  wtd_std_atomic_mass              21197 non-null  float64
 11  mean_fie                         21197 n

In [ ]:
target_colum = 'critical_temp'
X = df.drop(columns=[target_colum], errors='ignore')
y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lr_model = LinearRegression()

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
lr_model.fit(X_train, y_train)

# 3. Tiến hành dự đoán cân nặng
preds = lr_model.predict(X_test)

# 4. Đánh giá mô hình bằng R2-score, RMSE, MAE
r2_score_all = round(r2_score(y_test, preds), 3)
rmse_all = round(np.sqrt(mean_squared_error(y_test, preds)), 3)
mae_all = round(mean_absolute_error(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng Linear Regression: {r2_score_all}")
print(f"Chỉ số RMSE khi sử dụng Linear Regression: {rmse_all}")
print(f"Chỉ số MAE khi sử dụng Linear Regression: {mae_all}")

Chỉ số R2-score khi sử dụng Linear Regression: 0.735
Chỉ số RMSE khi sử dụng Linear Regression: 17.665
Chỉ số MAE khi sử dụng Linear Regression: 13.289


In [ ]:
# --- 1. Cấu hình và chạy Boruta ---
# BorutaPy yêu cầu mảng NumPy (.values) để tránh lỗi index
X_train_np = X_train.values
y_train_np = y_train.values

# Giữ nguyên Random Forest làm mô hình nền để Boruta tính toán feature_importances_
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
boruta_selector = BorutaPy(estimator=rf, n_estimators='auto', verbose=0, random_state=42, max_iter=100)
boruta_selector.fit(X_train_np, y_train_np)

# --- 2. Lọc và huấn luyện lại mô hình ---
# Lọc lấy các tính năng quan trọng được Boruta xác nhận
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test.values)

#Khởi tạo và huấn luyện mô hình Linear Regression trên tập dữ liệu đã lọc
lr_final = LinearRegression()
lr_final.fit(X_train_filtered, y_train_np)

# --- 3. Dự đoán và đánh giá điểm R2 ---
# Sử dụng mô hình Linear Regression vừa train để dự đoán
preds_boruta = lr_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test.values, preds_boruta), 3)
rsme_boruta = round(np.sqrt(mean_squared_error(y_test.values, preds_boruta)), 3)
mae_boruta = round(mean_absolute_error(y_test.values, preds_boruta), 3)

# --- 4. Trích xuất danh sách tính năng được chọn ---
feature_names = X_train.columns
confirmed_features = feature_names[boruta_selector.support_].tolist()

# --- 5. Xuất kết quả báo cáo ---
print(f"{'KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_boruta}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rsme_boruta}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_boruta}")
print(f"• Số lượng tính năng được giữ lại   : {len(confirmed_features)}/{len(feature_names)}")
print(f"• Các tính năng được giữ lại:\n  {confirmed_features}")
print("="*60)

  KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION  
• Chỉ số R2-score (Linear Regression) : 0.713
• Chỉ số RMSE (Linear Regression)    : 18.389
• Chỉ số MAE (Linear Regression)     : 14.073
• Số lượng tính năng được giữ lại   : 52/81
• Các tính năng được giữ lại:
  ['wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'wtd_mean_fie', 'gmean_fie', 'wtd_entropy_fie', 'range_fie', 'wtd_range_fie', 'wtd_mean_atomic_radius', 'range_atomic_radius', 'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius', 'mean_Density', 'gmean_Density', 'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density', 'std_Density', 'mean_ElectronAffinity', 'gmean_ElectronAffinity', 'wtd_gmean_ElectronAffinity', 'entropy_ElectronAffinity', 'wtd_entropy_ElectronAffinity', 'range_ElectronAffinity', 'wtd_range_ElectronAffinity', 'wtd_std

In [ ]:
# --- 1. Chuẩn hóa dữ liệu (Feature Scaling) ---
# Bước này bắt buộc phải có vì Lasso rất nhạy cảm với scale của dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Khởi tạo và chạy LassoCV để SÀNG LỌC TÍNH NĂNG ---
lasso = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, random_state=42, n_jobs=-1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# Trích xuất danh sách tên các tính năng được Lasso giữ lại (Coef != 0)
feature_names = X_train.columns
kept_features = feature_names[lasso.coef_ != 0].tolist()
dropped_features = feature_names[lasso.coef_ == 0].tolist()

# --- 3. Lọc lại dữ liệu theo các tính năng được chọn ---
# Lấy vị trí index của các tính năng được giữ lại để lọc trên mảng numpy đã scaled
kept_features_idx = [X_train.columns.get_loc(col) for col in kept_features]
X_train_filtered = X_train_scaled[:, kept_features_idx]
X_test_filtered = X_test_scaled[:, kept_features_idx]

# --- 4. Huấn luyện mô hình LINEAR REGRESSION trên các tính năng đã lọc ---
lr_final = LinearRegression(n_jobs=-1)
lr_final.fit(X_train_filtered, y_train)

# --- 5. Dự đoán và tính toán các chỉ số đánh giá ---
preds_lr = lr_final.predict(X_test_filtered)
r2_score_lr = round(r2_score(y_test, preds_lr), 3)
rmse_lr = round(np.sqrt(mean_squared_error(y_test, preds_lr)), 3)
mae_lr = round(mean_absolute_error(y_test, preds_lr), 3)

# --- 6. Xuất kết quả báo cáo ---
print("="*60)
print(f"{'KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print("="*60)
print(f"• Alpha tối ưu của Lasso             : {round(lasso.alpha_, 5)}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_lr}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rmse_lr}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_lr}")
print(f"• Số lượng tính năng bị loại bỏ      : {len(dropped_features)}/{len(feature_names)}")
print(f"• Số lượng tính năng được giữ lại    : {len(kept_features)}/{len(feature_names)}")
print("-"*60)
if dropped_features:
    print(f"• Các tính năng bị loại bỏ (Lasso Coef = 0):\n  {dropped_features}\n")
print(f"• Các tính năng được chọn để chạy Linear Regression:\n  {kept_features}")
print("="*60)

d:\C++\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.667558e+03, tolerance: 1.582e+03
  model = cd_fast.enet_coordinate_descent_gram(
d:\C++\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.738323e+03, tolerance: 1.582e+03
  model = cd_fast.enet_coordinate_descent_gram(
d:\C++\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:825: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.894643e+03, tolerance: 1.582e+03
  mo

 KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION  
• Alpha tối ưu của Lasso             : 0.0001
• Chỉ số R2-score (Linear Regression) : 0.735
• Chỉ số RMSE (Linear Regression)    : 17.65
• Chỉ số MAE (Linear Regression)     : 13.27
• Số lượng tính năng bị loại bỏ      : 0/81
• Số lượng tính năng được giữ lại    : 81/81
------------------------------------------------------------
• Các tính năng được chọn để chạy Linear Regression:
  ['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie', 'wtd_mean_fie', 'gmean_fie', 'wtd_gmean_fie', 'entropy_fie', 'wtd_entropy_fie', 'range_fie', 'wtd_range_fie', 'std_fie', 'wtd_std_fie', 'mean_atomic_radius', 'wtd_mean_atomic_radius', 'gmean_atomic_radius', 'wtd_gmean_atomic_radius', 'entropy_atomic_radius', 'wtd_entropy_atomic_radius', '

d:\C++\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.426951e+06, tolerance: 1.991e+03
  model = cd_fast.enet_coordinate_descent(
